In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os
import sys

import os; sys.path.append(os.path.abspath('..'))
import sys; import os; sys.path.append(os.path.abspath('..'))
from config import OUTPUTS_FIGURES_DIR

def run_mkz_analysis(df_test, y_test, lgbm_preds):
    os.makedirs(OUTPUTS_FIGURES_DIR, exist_ok=True)
    
    # 10a. Zone effect on accuracy
    bins = list(range(0, 105, 5))
    bins[-1] = np.inf
    labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]
    labels[-1] = "100+"
    
    df_test['dist_bin'] = pd.cut(df_test['dist_to_dest_km'], bins=bins, labels=labels, right=False)
    
    results = []
    for bin_label in labels:
        mask = df_test['dist_bin'] == bin_label
        if mask.sum() > 0:
            y_t = y_test[mask]
            y_p = lgbm_preds[mask]
            rmse = np.sqrt(mean_squared_error(y_t, y_p))
            mae = mean_absolute_error(y_t, y_p)
            w1 = (np.abs(y_t - y_p) <= 1.0).mean() * 100
            results.append({'bin': bin_label, 'RMSE': rmse, 'MAE': mae, 'Within_1hr_pct': w1})
            
    df_res = pd.DataFrame(results)
    
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=df_res, x='bin', y='RMSE', marker='o')
    plt.xticks(rotation=45)
    plt.title('RMSE vs Distance to Destination (Micro-Kinematic Zone Effect)')
    plt.xlabel('Distance Bin (km)')
    plt.ylabel('RMSE (hours)')
    plt.axvline(x=9.5, color='red', linestyle='--', alpha=0.5, label='MKZ Boundary (50km)') # 50km is at index 10
    plt.legend()
    plt.savefig(os.path.join(OUTPUTS_FIGURES_DIR, 'mkz_accuracy_vs_distance.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # 10b. Dominant feature analysis inside the zone
    features = [c for c in df_test.columns if c not in ['MMSI', 'BaseDateTime', 'VesselName', 'ETA_hours', 'arrived', 'arrival_timestamp', 'dist_bin']]
    features = [c for c in features if pd.api.types.is_numeric_dtype(df_test[c])]
    
    mkz_mask = df_test['dist_to_dest_km'] <= 50
    X_mkz = df_test.loc[mkz_mask, features]
    y_mkz = y_test[mkz_mask]
    
    if len(X_mkz) > 100:
        model_mkz = lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        model_mkz.fit(X_mkz, y_mkz)
        
        imp_mkz = pd.DataFrame({'feature': features, 'importance': model_mkz.feature_importances_})
        imp_mkz['type'] = 'Inside MKZ (<50km)'
        imp_mkz['importance'] = imp_mkz['importance'] / imp_mkz['importance'].sum()
        
        # We assume full model importance is saved or we re-calc a quick one
        model_full = lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        model_full.fit(df_test[features], y_test)
        
        imp_full = pd.DataFrame({'feature': features, 'importance': model_full.feature_importances_})
        imp_full['type'] = 'Global'
        imp_full['importance'] = imp_full['importance'] / imp_full['importance'].sum()
        
        df_imp = pd.concat([imp_mkz, imp_full])
        
        # Get top 10 from mkz
        top10 = imp_mkz.sort_values('importance', ascending=False).head(10)['feature']
        df_imp_top = df_imp[df_imp['feature'].isin(top10)]
        
        plt.figure(figsize=(12, 8))
        sns.barplot(data=df_imp_top, y='feature', x='importance', hue='type')
        plt.title('Feature Importance Shift (Inside MKZ vs Global)')
        plt.savefig(os.path.join(OUTPUTS_FIGURES_DIR, 'mkz_feature_shift.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
    # 10c. Velocity stability analysis
    errors = np.abs(y_test - lgbm_preds)
    if 'SOG_rolling_std_3' in df_test.columns:
        corr = df_test['SOG_rolling_std_3'].corr(pd.Series(errors, index=df_test.index))
        print(f"Pearson correlation between SOG_rolling_std_3 and absolute error: {corr:.4f}")

if __name__ == "__main__":
    # In practice this would load the test set and predictions.
    pass

